In [ ]:
# @title 🖥️ **WebUI Installer** {"display-mode":"form"}
# @markdown ### Step 1 — Pick your WebUI
Webui = 'Forge-Neo' # @param ["A1111", "Forge", "ReForge", "ReForge-old", "Forge-Classic", "Forge-Neo", "ComfyUI", "SwarmUI"]
# @markdown ---
# @markdown ### Step 2 — API Keys
# @markdown > 🔑 Get Civitai key → [Get Civitai key](https://civitai.com/user/account)
Civitai__Key = '' # @param {type:"string", placeholder:"Your Civitai API Key (required)"}
# @markdown > 🤗 Get HF token → [Get Hugging Face token](https://huggingface.co/settings/tokens)
HF_Read_Token = '' # @param {type:"string", placeholder:"Your Huggingface READ Token (optional)"}
# @markdown ---
# @markdown ### Step 3 — Google Drive *(optional — for persistent storage)*
# @markdown > If **Yes**, models are saved to `MyDrive/Segsmaker/` and survive session resets.
Mount_GDrive = 'No' # @param ["Yes", "No"]

from pathlib import Path
import json
import os
import shlex
import subprocess

SEGS_REPO = 'https://github.com/N3iKos/segsmaker-fast'
SEG_DRIVE_ROOT = Path('/content/drive/MyDrive/Segsmaker')
SEG_DRIVE_ENABLED = Mount_GDrive == 'Yes'

if SEG_DRIVE_ENABLED:
    from google.colab import drive
    drive.mount('/content/drive')
    SEG_DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

!curl -sLo /content/setup.py https://github.com/N3iKos/segsmaker-fast/raw/main/script/KC/setup.py
%run /content/setup.py --webui="$Webui" --civitai_key="$Civitai__Key" --hf_read_token="$HF_Read_Token"

def seg_drive_dir(name):
    folder = SEG_DRIVE_ROOT / name
    if SEG_DRIVE_ENABLED:
        folder.mkdir(parents=True, exist_ok=True)
    return folder

def seg_link_dir(name, runtime_path):
    if not SEG_DRIVE_ENABLED or runtime_path is None:
        return
    runtime_path = Path(runtime_path)
    runtime_path.mkdir(parents=True, exist_ok=True)
    link = runtime_path / f'drive-{name}'
    target = seg_drive_dir(name)
    if link.exists() or link.is_symlink():
        return
    try:
        link.symlink_to(target, target_is_directory=True)
        print(f'[Drive] linked {link} -> {target}')
    except Exception as exc:
        print(f'[Drive] link skipped for {name}: {exc}')

def seg_download_many(items, target_dir, drive_name, parallel=False, max_workers=3, load_from_drive=False):
    target_dir = Path(target_dir) if target_dir else None
    if target_dir is None:
        print(f'[skip] target path for {drive_name} is not available for this WebUI')
        return []
    target_dir.mkdir(parents=True, exist_ok=True)
    drive_dir = seg_drive_dir(drive_name) if (SEG_DRIVE_ENABLED and load_from_drive) else None
    return download_many(
        items,
        target_dir=str(target_dir),
        parallel=parallel,
        max_workers=max_workers,
        load_from_drive=bool(load_from_drive and SEG_DRIVE_ENABLED),
        drive_dir=str(drive_dir) if drive_dir else None,
    )

def seg_clone_repos(urls, target_dir, parallel=False, max_workers=3):
    urls = [url.strip() for url in urls if url.strip()]
    if not urls:
        print('  no extension/custom-node URLs')
        return []
    target_dir = Path(target_dir) if target_dir else None
    if target_dir is None:
        print('[skip] extensions path is not available for this WebUI')
        return []
    target_dir.mkdir(parents=True, exist_ok=True)

    def clone_one(url):
        command = shlex.split(url if url.startswith('git clone ') else f'git clone {url}')
        subprocess.run(command, cwd=str(target_dir), check=True)
        return command[-1]

    if not parallel or len(urls) == 1:
        results = []
        for index, url in enumerate(urls, 1):
            result = clone_one(url)
            print(f'  [{index:>2}/{len(urls)}] ? {result}')
            results.append(result)
        return results

    from concurrent.futures import ThreadPoolExecutor, as_completed
    results = []
    with ThreadPoolExecutor(max_workers=max(1, min(int(max_workers or 1), len(urls)))) as executor:
        futures = {executor.submit(clone_one, url): url for url in urls}
        for index, future in enumerate(as_completed(futures), 1):
            try:
                result = future.result()
                print(f'  [{index:>2}/{len(urls)}] ? {result}')
                results.append(result)
            except Exception as exc:
                print(f'  [{index:>2}/{len(urls)}] ? {futures[future]}: {exc}')
    return results

if SEG_DRIVE_ENABLED:
    for name, runtime_path in {
        'checkpoint': globals().get('CKPT'),
        'lora': globals().get('LORA'),
        'vae': globals().get('VAE'),
        'embeddings': globals().get('Embeddings'),
        'upscalers': globals().get('Upscalers'),
        'unet': globals().get('UNET'),
        'clip': globals().get('CLIP'),
        'text_encoder': globals().get('TE'),
    }.items():
        seg_link_dir(name, runtime_path)

    output_dir = SEG_DRIVE_ROOT / {'ComfyUI': 'comfyui-output', 'SwarmUI': 'swarmui-output'}.get(Webui, 'output')
    output_dir.mkdir(parents=True, exist_ok=True)
    try:
        if WebUI_Output.exists() and not WebUI_Output.is_symlink():
            print(f'[Drive] output path already exists, leaving it unchanged: {WebUI_Output}')
        elif not WebUI_Output.exists():
            WebUI_Output.symlink_to(output_dir, target_is_directory=True)
    except Exception as exc:
        print(f'[Drive] output link skipped: {exc}')


In [ ]:
# @title 📥 **Model Downloader - 5 Checkpoint + 5 LoRA + VAE** {"display-mode":"form"}
# @markdown ### 🗃️ Checkpoints
Checkpoint_1 = '' # @param {type:"string", placeholder:"URL Checkpoint 1"}
Checkpoint_2 = '' # @param {type:"string", placeholder:"URL Checkpoint 2"}
Checkpoint_3 = '' # @param {type:"string", placeholder:"URL Checkpoint 3"}
Checkpoint_4 = '' # @param {type:"string", placeholder:"URL Checkpoint 4"}
Checkpoint_5 = '' # @param {type:"string", placeholder:"URL Checkpoint 5"}
# @markdown ---
# @markdown ### 🎨 LoRA
Lora_1 = '' # @param {type:"string", placeholder:"URL Lora 1"}
Lora_2 = '' # @param {type:"string", placeholder:"URL Lora 2"}
Lora_3 = '' # @param {type:"string", placeholder:"URL Lora 3"}
Lora_4 = '' # @param {type:"string", placeholder:"URL Lora 4"}
Lora_5 = '' # @param {type:"string", placeholder:"URL Lora 5"}
# @markdown ---
# @markdown ### 🎛️ VAE
VAE_URL = '' # @param {type:"string", placeholder:"URL VAE or leave empty"}
# @markdown ---
# @markdown ### ⚡ Speed Options
Parallel_Download = True # @param {type:"boolean"}
Max_Workers = 3 # @param {type:"slider", min:1, max:10, step:1}

ckpt_items = [Checkpoint_1, Checkpoint_2, Checkpoint_3, Checkpoint_4, Checkpoint_5]
lora_items = [Lora_1, Lora_2, Lora_3, Lora_4, Lora_5]
vae_items = [VAE_URL]

ckpt_urls = [u.strip() for u in ckpt_items if u.strip()]
lora_urls = [u.strip() for u in lora_items if u.strip()]
vae_urls = [u.strip() for u in vae_items if u.strip()]

if ckpt_urls:
    print('📥 Downloading Checkpoints...')
    seg_download_many(ckpt_urls, globals().get('CKPT'), 'checkpoint', Parallel_Download, Max_Workers, True)
    
if lora_urls:
    print('📥 Downloading LoRAs...')
    seg_download_many(lora_urls, globals().get('LORA'), 'lora', Parallel_Download, Max_Workers, True)
    
if vae_urls:
    print('📥 Downloading VAE...')
    seg_download_many(vae_urls, globals().get('VAE'), 'vae', Parallel_Download, Max_Workers, True)

if not any([ckpt_urls, lora_urls, vae_urls]):
    print('ℹ️ No URLs provided.')
else:
    print('✅ Download batch completed!')


In [ ]:
# @title 🛠️ **Extra Assets - Extensions, Embeddings, Upscalers** {"display-mode":"form"}
# @markdown ### 🔌 Extensions / ComfyUI Custom Nodes
Extension_1 = '' # @param {type:"string", placeholder:"git clone URL or leave empty"}
Extension_2 = '' # @param {type:"string", placeholder:"git clone URL or leave empty"}
Extension_3 = '' # @param {type:"string", placeholder:"git clone URL or leave empty"}
Extension_4 = '' # @param {type:"string", placeholder:"git clone URL or leave empty"}
Extension_5 = '' # @param {type:"string", placeholder:"git clone URL or leave empty"}
# @markdown ---
# @markdown ### 🖼️ Embeddings
Embedding_1 = '' # @param {type:"string", placeholder:"URL or leave empty"}
Embedding_2 = '' # @param {type:"string", placeholder:"URL or leave empty"}
Embedding_3 = '' # @param {type:"string", placeholder:"URL or leave empty"}
# @markdown ---
# @markdown ### 🔬 Upscalers
Upscaler_1 = '' # @param {type:"string", placeholder:"URL or leave empty"}
Upscaler_2 = '' # @param {type:"string", placeholder:"URL or leave empty"}
Upscaler_3 = '' # @param {type:"string", placeholder:"URL or leave empty"}
# @markdown ---
# @markdown ### ⚡ Speed Options
Assets_Parallel_Download = True # @param {type:"boolean"}
Assets_Max_Workers = 3 # @param {type:"slider", min:1, max:10, step:1}

from pathlib import Path

ext_items = [Extension_1, Extension_2, Extension_3, Extension_4, Extension_5]
emb_items = [Embedding_1, Embedding_2, Embedding_3]
ups_items = [Upscaler_1, Upscaler_2, Upscaler_3]

ext_urls = [u.strip() for u in ext_items if u.strip()]
emb_urls = [u.strip() for u in emb_items if u.strip()]
ups_urls = [u.strip() for u in ups_items if u.strip()]

if ext_urls:
    print('⚡ Cloning extensions...')
    seg_clone_repos(ext_urls, globals().get('Extensions'), Assets_Parallel_Download, Assets_Max_Workers)
    
if emb_urls:
    print('📥 Downloading Embeddings...')
    seg_download_many(emb_urls, globals().get('Embeddings'), 'embeddings', Assets_Parallel_Download, Assets_Max_Workers, True)
    
if ups_urls:
    print('📥 Downloading Upscalers...')
    ups_dest = globals().get('Upscalers') or (Path(globals().get('WebUI')) / 'models/ESRGAN')
    seg_download_many(ups_urls, ups_dest, 'upscalers', Assets_Parallel_Download, Assets_Max_Workers, True)

if not any([ext_urls, emb_urls, ups_urls]):
    print('ℹ️ No URLs provided.')
else:
    print('✅ Assets process completed!')


In [ ]:
# @title ⚡ **FLUX Model Downloader** {"display-mode":"form"}
# @markdown ### Select FLUX Variant
FLUX_Variant = 'FLUX.1-schnell (Fast, 4-step)' # @param ["FLUX.1-schnell (Fast, 4-step)", "FLUX.1-dev (Quality, 20-step)"]
# @markdown ---
# @markdown ### Component URLs
FLUX_Unet = '' # @param {type:"string", placeholder:"Custom Unet URL or leave empty"}
FLUX_Clip_L = '' # @param {type:"string", placeholder:"Custom Clip L URL or leave empty"}
FLUX_T5XXL = '' # @param {type:"string", placeholder:"Custom T5XXL URL or leave empty"}
FLUX_VAE = '' # @param {type:"string", placeholder:"Custom VAE URL or leave empty"}
# @markdown ---
# @markdown ### ⚡ Speed Options
Parallel_FLUX_Download = True # @param {type:"boolean"}
FLUX_Max_Workers = 2 # @param {type:"slider", min:1, max:6, step:1}

FLUX_DEFAULTS = {
    'FLUX.1-schnell (Fast, 4-step)': {
        'unet': 'https://huggingface.co/Kijai/flux-fp8/resolve/main/flux1-schnell-fp8.safetensors',
        'clip_l': 'https://huggingface.co/comfyanonymous/flux_text_encoders/resolve/main/clip_l.safetensors',
        't5xxl': 'https://huggingface.co/comfyanonymous/flux_text_encoders/resolve/main/t5xxl_fp8_e4m3fn.safetensors',
        'vae': 'https://huggingface.co/black-forest-labs/FLUX.1-schnell/resolve/main/ae.safetensors'
    },
    'FLUX.1-dev (Quality, 20-step)': {
        'unet': 'https://huggingface.co/Kijai/flux-fp8/resolve/main/flux1-dev-fp8.safetensors',
        'clip_l': 'https://huggingface.co/comfyanonymous/flux_text_encoders/resolve/main/clip_l.safetensors',
        't5xxl': 'https://huggingface.co/comfyanonymous/flux_text_encoders/resolve/main/t5xxl_fp8_e4m3fn.safetensors',
        'vae': 'https://huggingface.co/black-forest-labs/FLUX.1-schnell/resolve/main/ae.safetensors'
    }
}

preset = FLUX_DEFAULTS[FLUX_Variant]
unet_url = FLUX_Unet.strip() or preset['unet']
clip_l_url = FLUX_Clip_L.strip() or preset['clip_l']
t5xxl_url = FLUX_T5XXL.strip() or preset['t5xxl']
vae_url = FLUX_VAE.strip() or preset['vae']

flux_unet_dir = globals().get('UNET') or globals().get('CKPT')
flux_clip_dir = globals().get('CLIP')
flux_t5_dir = globals().get('TE') or globals().get('CLIP')
flux_vae_dir = globals().get('VAE')

print(f'⚡ Downloading FLUX components ({FLUX_Variant})...')
seg_download_many([unet_url], flux_unet_dir, 'flux-unet', Parallel_FLUX_Download, FLUX_Max_Workers, True)
seg_download_many([clip_l_url], flux_clip_dir, 'flux-clip', Parallel_FLUX_Download, FLUX_Max_Workers, True)
seg_download_many([t5xxl_url], flux_t5_dir, 'flux-text-encoder', Parallel_FLUX_Download, FLUX_Max_Workers, True)
seg_download_many([vae_url], flux_vae_dir, 'flux-vae', Parallel_FLUX_Download, FLUX_Max_Workers, True)

print('\n✅ FLUX models processed!')


In [ ]:
''' Controlnet '''
%run $Controlnet_Widget

In [ ]:
# @title 🚀 **Launcher WebUI** {"display-mode":"form"}
# @markdown Select the same WebUI that you installed in the first cell.
Software = 'Forge-Neo' # @param ["A1111", "Forge", "ReForge", "ReForge-old", "Forge-Classic", "Forge-Neo", "ComfyUI", "SwarmUI"]
# @markdown ---
# @markdown ### Tunnel Tokens
# @markdown > Optional network tunnels
Ngrok_Token = '' # @param {type:"string"}
Zrok_Token = '' # @param {type:"string"}
# @markdown ---
# @markdown ### Pengaturan Tambahan
Extra_Args = '' # @param {type:"string"}
Skip_ComfyUI_Check = False # @param {type:"boolean"}
Skip_Widget = False # @param {type:"boolean"}

import shlex
import json

print(f'📦 Launching {Software}...')

installed = None
try:
    home_path = globals().get('HOMEPATH') or Path.home()
    mark_path = Path(home_path) / 'gutris1/marking.json'
    if mark_path.exists():
        installed = json.loads(mark_path.read_text()).get('ui')
except Exception:
    pass
    
if not installed:
    installed = globals().get('Webui')
    
if installed and Software != installed:
    print(f'[warning] Installed WebUI is {installed}, but launcher selection is {Software}. The installed WebUI will be used by segsmaker.py.')

args = []
if Skip_ComfyUI_Check:
    args.append('--skip-comfyui-check')
if Skip_Widget:
    args.append('--skip-widget')
if Ngrok_Token.strip():
    args.append(f'--N={Ngrok_Token.strip()}')
if Zrok_Token.strip():
    args.append(f'--Z={Zrok_Token.strip()}')
if Extra_Args.strip():
    args.extend(shlex.split(Extra_Args.strip()))
    
webui_path = globals().get('WebUI')
if not webui_path:
    from pathlib import Path
    webui_path = Path('/content') / Software
    
import os
os.chdir(str(webui_path))

run_line = 'segsmaker.py ' + ' '.join(shlex.quote(arg) for arg in args)
print(f'🚀 Running command: %run {run_line}')
get_ipython().run_line_magic('run', run_line)
